<a href="https://colab.research.google.com/github/coitloz88/govcrawler/blob/master/%EA%B5%AD%ED%86%A0%EA%B5%90%ED%86%B5%EB%B6%80_%EB%B3%B4%EB%8F%84%EC%9E%90%EB%A3%8C_%EC%9E%90%EB%8F%99%ED%99%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 간략한 Prompt 작성요령

우리가 단순히 "다운로드해줘"라고만 하면 AI는 코드를 짜주지만, 실행해보면 에러가 나거나 빈 파일만 생깁니다. 왜 그럴까요? **AI는 우리의 작업 환경(Colab, Replit)과 대상 사이트(국토부)의 특수성을 모르기 때문**입니다.

다음 4가지 항목은 반드시 프롬프트에 포함되어야 합니다.

---

#### (1) 도구의 변경: BeautifulSoup 대신 Selenium 사용

보통 웹 크롤링이라고 하면 `BeautifulSoup` 라이브러리를 먼저 떠올립니다. 하지만 국토부 같은 공공기관 사이트나 최신 웹사이트들은 **"동적 웹페이지(Javascript)"** 기술을 사용합니다. 쉽게 말해, 우리가 눈으로 볼 때는 파일 목록이 보이지만, 코드(`requests`)로만 접속하면 **파일 목록이 로딩되기 전의 빈 껍데기**만 가져오게 됩니다.

그래서 **"사람이 브라우저를 켜고 기다렸다가 클릭하는 동작"**을 그대로 흉내 낼 수 있는 `Selenium` 라이브러리를 사용해야 합니다.

**[AI에게 입력할 핵심 프롬프트]**

> **"이 사이트는 자바스크립트로 파일 목록을 나중에 불러오는 동적 페이지야. `requests`나 `BeautifulSoup` 대신 반드시 `Selenium`을 사용해서 페이지가 완전히 로딩된 후(`WebDriverWait`) 데이터를 수집하도록 코드를 짜줘."**

---

#### (2) 환경 설정: Colab/Replit(Linux) 맞춤형 크롬 설치

Selenium은 **'크롬 브라우저'**를 조종하는 도구입니다. 그런데 여러분이 실습하는 **Google Colab**이나 **Replit**은 윈도우(Windows)가 아니라 **리눅스(Linux)**라는 서버 컴퓨터 환경입니다. 여기에는 **크롬 브라우저가 깔려있지 않습니다.**

AI에게 그냥 Selenium 코드를 짜달라고 하면, AI는 당연히 크롬이 깔려있다고 가정하고 코드를 짭니다. 결과는 당연히 `WebDriverException` 에러겠죠? 따라서 **"브라우저부터 설치하고 시작해"**라고 알려줘야 합니다.

**[AI에게 입력할 핵심 프롬프트]**

> **"나는 지금 Google Colab(Linux 환경)에서 작업 중이라 크롬 브라우저가 없어.
> wget을 이용해 크롬을 구글로 직접 다운로드를 받고 webdriver 매니저를 통해 chrome driver를 설치하고 버전 관리를 해줘. subprocess 방식을 써주어야 더 안정적이니 그걸 꼭 고려해줘

---

#### (3) 언어의 장벽: 인코딩 문제 (EUC-KR vs UTF-8)

컴퓨터는 '한글'을 숫자로 변환해서 이해하는데, 이 규칙을 **인코딩**이라고 합니다.  `EUC-KR`도 많이 사용되지만, 요즘은 전 세계 표준인 `UTF-8`을 씁니다.

문제는 우리가 검색어(예: "주택통계")를 서버에 보낼 때 발생합니다. 국토부 서버는 **UTF-8**로 된 검색어를 기다리는데, AI가 습관적으로(또는 사이트가 오래돼 보여서) **EUC-KR**로 변환해서 보내면, 서버는 "쮀턕톙계" 같은 외계어로 인식하고 **"검색 결과 0건"**을 돌려줍니다.

**[AI에게 입력할 핵심 프롬프트]**

> **"검색어('주택통계')를 URL에 넣을 때는 반드시 `UTF-8` 방식으로 인코딩해 줘. (`urllib.parse.quote` 사용). 만약 EUC-KR로 보내면 서버가 인식을 못 해서 검색 결과가 안 나오니까 주의해 줘."**

---

#### (4) 다운로드 전략: 하이브리드 방식 & 중복 방지

Selenium은 버튼을 클릭하는 데는 좋지만, 파일을 다운로드할 때는 속도가 느리고 저장 경로 설정이 까다롭습니다. 반면 `requests`는 다운로드가 빠르고 간편합니다.

그래서 우리는 **"검색은 똑똑한 Selenium에게 맡기고, 다운로드라는 단순 노동은 빠른 requests에게 맡기는"** 하이브리드 방식을 쓸 겁니다. 이때 중요한 건 Selenium이 로그인하면서 얻은 **쿠키(입장권)**를 requests에게 복사해주는 것입니다.

또한, 게시판에는 진짜 파일 외에도 '미리보기', '뷰어', '썸네일' 같은 가짜 링크들이 많습니다. 이를 거르지 않으면 똑같은 파일을 3~4번씩 중복해서 받게 됩니다.

**[AI에게 입력할 핵심 프롬프트]**

> **"1. 파일 다운로드 시에는 Selenium으로 클릭하지 말고, 쿠키(Session Cookie)를 `requests` 세션에 복사해서 다운로드하는 '하이브리드 방식'을 써줘.
> 2. 파일명이나 링크에 'viewer', '미리보기'가 포함된 건 제외하고, 진짜 `.pdf` 확장자를 가진 파일만 중복 없이(`set` 자료구조 사용) 다운로드해 줘."**

## 2. prmopt 예시



**1. 역할 지정**
너는 지금부터 '파이썬 웹 크롤링 전문가'야. 구글 코랩(Google Colab) 환경에서 실행할 수 있는 완벽한 코드를 작성해 줘.

**2. 목표**
국토교통부 보도자료 게시판에서 "주택통계"라는 키워드로 검색된 게시글들의 첨부파일(PDF)을 내 컴퓨터(코랩 폴더)로 자동으로 다운로드하고 싶어.


**3. 요구사항**

* **환경 해결:** 코랩에는 크롬 브라우저가 안 깔려 있어. 코드가 실행될 때 **최신 크롬과 드라이버를 자동으로 설치**해서 버전 오류가 안 나게 해줘. (wget을 이용해 크롬을 구글로 직접 다운로드를 받고 webdriver 매니저를 통해 chrome driver를 설치하고 버전 관리를 해줘,deb로 직접 그리고 subprocess 방식을 써주어야 더 안정적이니 그걸 꼭 고려해줘)

* **하이브리드 방식:**
* 게시글 목록을 찾고 클릭하는 건 **Selenium(브라우저 조종)**을 써줘. (자바스크립트 로딩 때문이야)
* 하지만 파일 다운로드는 **Requests(통신 라이브러리)**를 써줘. (Selenium으로 클릭하면 팝업 뜨고 느리니까, 쿠키만 복사해서 다운로드해)


* **검색어 처리:** 검색어("주택통계")를 URL에 넣을 때 한글이 깨지지 않도록 **UTF-8 인코딩**을 꼭 적용해 줘. 안 그러면 검색 결과가 0건 나와.
* **파일 필터링:** "미리보기"나 "뷰어" 링크는 다운받지 마. 오직 **진짜 .pdf 파일**만 중복 없이 저장해 줘.

일단 검색결과와 다운로드 결과가 0다 그러면 너가 직접 홈페이지에 접속해 검색결과가 어떻게 나오는지 조회하는 코드도 짜주고 그걸 너한테 전달할 수 있게 알림메시지도 해줘. (Debug.html파일 생성을 활용)


**4. 실행 정보**

* **사이트:** `https://www.molit.go.kr/USR/NEWS/m_71/lst.jsp`
* **기간:** 2024-12-01 ~ 2025-06-30
* **저장 폴더:** `molit_stats_files`

이 조건들을 모두 만족하는 파이썬 코드를 작성해 줘. 중간에 멈추지 않게 예외 처리도 꼼꼼히 해줘.

---


## 3. 생성코드 예시 (실제 사용시에는 각 기능별로 분류하셔도 좋습니다.)


In [ ]:
# ==========================================
# [설정] 필수 라이브러리 설치 확인
# ==========================================
import os
import subprocess

# 코랩/리눅스 환경에서 크롬/드라이버 없으면 설치 (여기서 오류가 자주 발생합니다. 잘 되지 않는 경우 이 예시코드를 복사해서 사용해주세요!!!!)
if not os.path.exists("/usr/bin/google-chrome") and not os.path.exists("/usr/bin/chromium-browser"):
    print("🔄 크롬 브라우저 설치 중...")
    subprocess.run(["wget", "-q", "https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb"], check=True)
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "./google-chrome-stable_current_amd64.deb"], check=True)
    subprocess.run(["pip", "install", "selenium", "webdriver-manager", "requests"], check=True)

# ==========================================
# [실행] 중복 방지 + UTF-8 검색 다운로더
# ==========================================
import time
import re
import requests
import shutil
from urllib.parse import urljoin, quote
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 설정 영역 (원하는 값을 입력해주시면 됩니다!, 저희가 입력필드로 가져온 영역에 대한 value라고 볼 수 있습니다.)
START_DATE = "2024-12-01"
END_DATE = "2025-06-30"
SEARCH_KEYWORD = "주택통계"
DOWNLOAD_DIR = "molit_stats_files"

def download_molit_clean():
    if not os.path.exists(DOWNLOAD_DIR):
        os.makedirs(DOWNLOAD_DIR)

    print("🚀 브라우저 설정 중...")

    # 브라우저 옵션
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=options)
    except Exception as e:
        print(f"❌ 브라우저 초기화 실패: {e}")
        return

    try:
        base_url = "https://www.molit.go.kr"

        # [요청 반영] 검색어 인코딩을 무조건 'utf-8'로 고정
        encoded_keyword = quote(SEARCH_KEYWORD, encoding='utf-8')

        # 목록 페이지 URL
        list_url = (
            f"https://www.molit.go.kr/USR/NEWS/m_71/lst.jsp?"
            f"search_regdate_s={START_DATE}&"
            f"search_regdate_e={END_DATE}&"
            f"search={encoded_keyword}&"
            f"srch_type=sj&psize=50"
        )

        print(f"🔗 목록 이동 (UTF-8 검색): {list_url}")
        driver.get(list_url)

        # 목록 로딩 대기
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CSS_SELECTOR, "tbody tr")))

        articles = []
        rows = driver.find_elements(By.CSS_SELECTOR, "tbody > tr")

        for row in rows:
            try:
                link_el = row.find_element(By.CSS_SELECTOR, "a[href*='dtl.jsp']")
                title = link_el.text.strip()
                href = link_el.get_attribute("href")

                if "주택" in title and "통계" in title:
                    articles.append({'title': title, 'url': href})
            except:
                continue

        print(f"✅ 총 {len(articles)}개의 게시글을 찾았습니다.")

        # 상세 페이지 순회
        for idx, article in enumerate(articles, 1):
            print(f"\n[{idx}/{len(articles)}] 분석 중: {article['title']}")
            driver.get(article['url'])

            # 첨부파일 영역 대기
            try:
                time.sleep(2)
                WebDriverWait(driver, 5).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.bd_view"))
                )
            except:
                print("   ⚠️ 로딩 지연 (계속 진행)")

            # [중복 방지] 이 게시글에서 이미 다운로드한 파일명 저장소
            downloaded_in_this_post = set()

            # 링크 찾기
            links = driver.find_elements(By.TAG_NAME, "a")
            found = False

            for link in links:
                try:
                    file_href = link.get_attribute("href")
                    # href가 없거나 자바스크립트 호출이면 패스
                    if not file_href or "javascript" in file_href:
                        continue

                    # [중복 방지] 'viewer'나 '미리보기'가 주소/텍스트에 있으면 스킵
                    link_text = link.text.strip()
                    if "viewer" in file_href or "미리보기" in link_text:
                        continue

                    file_name = link.get_attribute("title")
                    if not file_name:
                        file_name = link_text

                    if file_name:
                        file_name = file_name.strip()

                    # 조건 확인 (주택 + pdf)
                    if "주택" in file_name and ".pdf" in file_name.lower():

                        # 파일명 정리
                        safe_name = re.sub(r'[\\/*?:"<>|]', "", file_name)
                        if not safe_name.lower().endswith('.pdf'): safe_name += ".pdf"

                        # [중복 방지] 이미 처리한 파일명이면 건너뜀
                        if safe_name in downloaded_in_this_post:
                            continue

                        # 다운로드 처리
                        print(f"   🔎 파일 발견: {safe_name}")
                        downloaded_in_this_post.add(safe_name) # 처리 목록에 추가

                        # Requests로 다운로드 (쿠키 복사)
                        session = requests.Session()
                        for cookie in driver.get_cookies():
                            session.cookies.set(cookie['name'], cookie['value'])

                        headers = {
                            "User-Agent": driver.execute_script("return navigator.userAgent;"),
                            "Referer": article['url']
                        }

                        save_path = os.path.join(DOWNLOAD_DIR, safe_name)

                        print(f"      ⬇️ 다운로드 중...")
                        file_res = session.get(file_href, headers=headers, stream=True)

                        if file_res.status_code == 200:
                            with open(save_path, 'wb') as f:
                                for chunk in file_res.iter_content(chunk_size=8192):
                                    f.write(chunk)
                            print(f"      💾 저장 완료")
                            found = True
                        else:
                            print(f"      ❌ 실패 (HTTP {file_res.status_code})")

                except Exception:
                    continue

            if not found:
                print("   ⚠️ PDF 파일을 찾지 못했습니다.")

    except Exception as e:
        print(f"❌ 실행 중 에러: {e}")

    finally:
        print("🏁 작업 완료")
        driver.quit()

if __name__ == "__main__":
    download_molit_clean()

🔄 크롬 브라우저 설치 중...
🚀 브라우저 설정 중...
🔗 목록 이동 (UTF-8 검색): https://www.molit.go.kr/USR/NEWS/m_71/lst.jsp?search_regdate_s=2024-12-01&search_regdate_e=2025-06-30&search=%EC%A3%BC%ED%83%9D%ED%86%B5%EA%B3%84&srch_type=sj&psize=50
✅ 총 6개의 게시글을 찾았습니다.

[1/6] 분석 중: ’25년 5월 주택통계
   🔎 파일 발견: 250630(석간)_’25년_5월_주택통계(주택정책과).pdf
      ⬇️ 다운로드 중...
      💾 저장 완료

[2/6] 분석 중: ‘25년 4월 주택통계
   🔎 파일 발견: 250530(석간)_‘25년_4월_주택_통계(주택정책과).pdf
      ⬇️ 다운로드 중...
      💾 저장 완료

[3/6] 분석 중: ‘25년 3월 주택통계
   🔎 파일 발견: 250429(석간)_25년_3월_주택_통계(주택정책과)_.pdf
      ⬇️ 다운로드 중...
      💾 저장 완료

[4/6] 분석 중: ‘25년 2월 주택통계
   🔎 파일 발견: 250331(석간)_‘25년_2월_주택_통계(주택정책과).pdf
      ⬇️ 다운로드 중...
      💾 저장 완료

[5/6] 분석 중: ‘25년 1월 주택통계
   🔎 파일 발견: 250228(석간)_‘25년_1월_주택_통계(주택정책과).pdf
      ⬇️ 다운로드 중...
      💾 저장 완료

[6/6] 분석 중: ‘24년 12월 주택통계
   🔎 파일 발견: 250205(석간)_‘24년_12월_주택_통계(주택정책과).pdf
      ⬇️ 다운로드 중...
      💾 저장 완료
🏁 작업 완료
